In [ ]:
import sqlite3
import pandas as pd
import tkinter as tk
from tkinter import messagebox
from tkinter import ttk

# ---------------- DATABASE ----------------

connection = sqlite3.connect("data_redundancy.db")
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT
)
""")

connection.commit()


# ---------------- ADD RECORD ----------------

def add_record():

    name = name_entry.get().strip()
    email = email_entry.get().strip()
    city = city_entry.get().strip()

    # Basic validation
    if name == "" or email == "" or city == "":
        messagebox.showwarning(
            "Missing Information",
            "Please enter Name, Email and City."
        )
        return

    # Check whether email already exists
    cursor.execute(
        "SELECT * FROM users WHERE email = ?",
        (email,)
    )

    existing = cursor.fetchone()

    if existing:
        result_label.config(
            text="REDUNDANT DATA ❌\nRecord already exists.",
            fg="red"
        )

        messagebox.showwarning(
            "Duplicate Found",
            "This record already exists in the database."
        )

    else:
        # Insert unique record
        cursor.execute(
            "INSERT INTO users (name, email, city) VALUES (?, ?, ?)",
            (name, email, city)
        )

        connection.commit()

        result_label.config(
            text="UNIQUE / VERIFIED DATA ✅\nRecord added successfully.",
            fg="green"
        )

        messagebox.showinfo(
            "Success",
            "Unique record added successfully!"
        )

        # Clear fields
        name_entry.delete(0, tk.END)
        email_entry.delete(0, tk.END)
        city_entry.delete(0, tk.END)

        show_database()


# ---------------- DISPLAY DATABASE ----------------

def show_database():

    for item in table.get_children():
        table.delete(item)

    df = pd.read_sql_query(
        "SELECT * FROM users",
        connection
    )

    for _, row in df.iterrows():

        table.insert(
            "",
            tk.END,
            values=(
                row["id"],
                row["name"],
                row["email"],
                row["city"]
            )
        )


# ---------------- GUI ----------------

root = tk.Tk()

root.title("Data Redundancy Removal System")
root.geometry("750x600")

title = tk.Label(
    root,
    text="DATA REDUNDANCY REMOVAL SYSTEM",
    font=("Arial", 18, "bold")
)

title.pack(pady=20)


# Name
tk.Label(
    root,
    text="Name:",
    font=("Arial", 12)
).pack()

name_entry = tk.Entry(
    root,
    width=40,
    font=("Arial", 12)
)

name_entry.pack(pady=5)


# Email
tk.Label(
    root,
    text="Email:",
    font=("Arial", 12)
).pack()

email_entry = tk.Entry(
    root,
    width=40,
    font=("Arial", 12)
)

email_entry.pack(pady=5)


# City
tk.Label(
    root,
    text="City:",
    font=("Arial", 12)
).pack()

city_entry = tk.Entry(
    root,
    width=40,
    font=("Arial", 12)
)

city_entry.pack(pady=5)


# Button
add_button = tk.Button(
    root,
    text="VALIDATE & ADD DATA",
    command=add_record,
    font=("Arial", 12, "bold"),
    padx=20,
    pady=10
)

add_button.pack(pady=15)


# Result
result_label = tk.Label(
    root,
    text="Enter data and click Validate",
    font=("Arial", 12, "bold")
)

result_label.pack(pady=10)


# Database table
columns = ("ID", "Name", "Email", "City")

table = ttk.Treeview(
    root,
    columns=columns,
    show="headings",
    height=8
)

for column in columns:
    table.heading(column, text=column)
    table.column(column, width=150)

table.pack(pady=15)


show_database()

root.mainloop()